# 표 데이터와 트리 모델 — 회귀, 랜덤 포레스트, 부스팅

> ⏱ 50분 · CPU로 충분

**목표:** 엑셀 같은 **표 데이터**로 숫자를 예측(회귀)합니다. 기준선(baseline)을 세우고, 결정 트리 → 랜덤 포레스트 → 그래디언트 부스팅으로 모델을 키워 가며 비교하는 실험 습관을 익힙니다.

> 실무에서 표 데이터는 지금도 딥러닝보다 **트리 기반 모델**이 더 잘 맞는 경우가 많습니다. 딥러닝이 압도적인 분야는 이미지·텍스트·음성처럼 "날것의" 데이터입니다.

## 데이터: 캘리포니아 집값

한 행이 한 지역, 열이 그 지역의 특징입니다. 맞혀야 할 값은 집값 중앙값(단위: 10만 달러)입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame                     # pandas DataFrame: 표 데이터를 다루는 표준 도구
print(df.shape)
df.head()

In [ ]:
print(df.describe().T[["mean", "min", "max"]])   # 열마다 범위가 제각각입니다
df.hist(bins=40, figsize=(12, 7)); plt.tight_layout(); plt.show()

모델을 만들기 전에 **데이터를 먼저 들여다보는 것**이 가장 가성비 좋은 작업입니다. 이상한 값(예: 방 개수 평균이 140개인 지역)이나 잘린 값(집값이 5.0에서 뭉쳐 있음)이 보이나요?

**코드 읽기**

- `fetch_california_housing(as_frame=True)` — `load_digits`와 달리 `fetch_`로 시작하는 함수는 인터넷에서 내려받습니다(약 400KB, 한 번만). `as_frame=True`를 주면 numpy 배열 대신 **열 이름이 붙은** pandas 표로 받습니다. 표 데이터에서는 열 이름이 곧 해석의 단서이므로 항상 이렇게 받습니다.
- `pandas`의 `DataFrame` — 엑셀 시트에 해당하는 자료형. `df.head()`(앞 5행), `df.describe()`(열별 통계), `df.hist()`(열별 히스토그램)처럼 **데이터를 훑어보는 도구**가 내장되어 있어서, 모델링 전 탐색은 거의 항상 pandas로 합니다.
- `df.describe().T[["mean", "min", "max"]]` — `describe()`는 열이 가로로 나열되어 읽기 어려우므로 `.T`로 뒤집고, 통계 중 세 개만 골라 봅니다. `AveRooms`의 최댓값 141처럼 **평균과 동떨어진 최댓값**은 이상치가 있다는 신호입니다.
- `df.hist(bins=40)` — 모든 숫자 열의 분포를 한 번에 그립니다. `MedHouseVal`의 오른쪽 끝이 5.0에서 솟아 있는 것은 데이터 수집 때 "5.0 이상은 5.0으로" 잘라 버렸기 때문입니다. 이런 사실은 모델이 아니라 **눈으로만** 발견할 수 있습니다.
- `plt.tight_layout()` — 작은 그림 8개의 제목이 서로 겹치지 않게 간격을 자동 조정합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(columns="MedHouseVal"), df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

**코드 읽기**

- `df.drop(columns="MedHouseVal")` — 정답 열을 뺀 나머지가 입력 X입니다. 정답을 입력에 실수로 남겨 두면 모델이 "정답을 보고 정답을 맞히는" 100점짜리 가짜 결과를 냅니다(데이터 누출). 표 데이터에서 가장 흔한 실수이니 `X.columns`를 한 번 출력해 확인하는 습관을 들이세요.
- 이번에는 `stratify`가 없습니다. 회귀 문제는 정답이 연속값이라 "클래스 비율"이라는 개념이 없기 때문입니다.
## 기준선부터 세운다

"무조건 평균값으로 답하기"보다 못한 모델은 의미가 없습니다. 가장 멍청한 방법의 점수를 먼저 알아 둬야 내 모델이 얼마나 좋은지 말할 수 있습니다.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

results = {}
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {"train MAE": mean_absolute_error(y_train, model.predict(X_train)),
                     "test MAE": mean_absolute_error(y_test, pred), "test R2": r2_score(y_test, pred)}
    print(f"{name:22s} train MAE {results[name]['train MAE']:.3f} | test MAE {results[name]['test MAE']:.3f} | R2 {results[name]['test R2']:.3f}")
    return model

evaluate("평균으로 찍기", DummyRegressor())
linear = evaluate("선형 회귀", LinearRegression())

- **MAE (평균 절대 오차):** 평균적으로 몇 (10만) 달러 틀리는지. 해석이 쉽습니다.
- **R²:** 1이면 완벽, 0이면 평균으로 찍는 것과 같음.

**코드 읽기**

- `DummyRegressor()` — 입력을 보지 않고 **항상 학습 데이터의 평균**을 답하는 "모델". 왜 굳이 모델 형태로 만드나: 다른 모델과 똑같이 `fit`/`predict`로 다룰 수 있어서 같은 `evaluate` 함수에 넣기만 하면 기준선이 나옵니다. 분류에는 `DummyClassifier`(최빈 클래스로 답함)가 있습니다.
- `evaluate(name, model)` 함수 — 학습 → 예측 → 지표 계산 → 딕셔너리에 저장 → 출력을 한 번에 합니다. 왜 함수로 묶나: 모델을 5개 비교할 텐데 같은 코드를 5번 복사하면 하나만 고쳐도 나머지를 빠뜨리게 됩니다. 실험 코드에서 "측정하는 부분"은 반드시 한 곳에 둡니다.
- `results[name] = {...}` — 결과를 딕셔너리에 모아 두면 나중에 `pd.DataFrame(results).T`로 한 번에 표를 만들 수 있습니다.
- `mean_absolute_error` vs `mean_squared_error` — MAE는 "평균 몇 달러 틀렸나"로 바로 읽히고, MSE는 큰 오차에 제곱으로 더 큰 벌을 줍니다. 사람에게 설명할 지표로는 MAE, 학습용 손실로는 MSE(미분이 매끄러움)를 쓰는 경우가 많습니다.
- `r2_score` — 단위와 무관한 0~1 척도라서 데이터셋이 달라도 "얼마나 잘 맞췄나"를 비교하기 좋습니다. 평균으로 찍으면 정확히 0이 나오는 것이 위 첫 줄에서 보입니다.
- `LinearRegression` — 트리 모델 전에 **선형 모델을 먼저** 돌리는 이유: 빠르고, 해석되고, 이보다 못한 복잡한 모델은 뭔가 잘못됐다는 뜻이기 때문입니다.

선형 회귀는 `집값 = w1×소득 + w2×집 나이 + … + b`입니다. 학습된 가중치를 보면 모델의 "생각"을 읽을 수 있습니다.

In [ ]:
print(pd.Series(linear.coef_, index=X.columns).round(3))

**코드 읽기**

- `pd.Series(값 배열, index=이름)` — 숫자 배열에 열 이름을 붙여 "어느 특징의 가중치인지" 읽을 수 있게 합니다. `MedInc`(소득)의 가중치가 +0.43이면 "소득이 1 늘면 집값이 4.3만 달러 오른다"로 읽습니다. `Latitude`, `Longitude`가 음수인 것은 북쪽·동쪽으로 갈수록 싸다는 뜻(캘리포니아 해안이 남서쪽). 이렇게 **모델을 읽을 수 있다**는 것이 선형 모델의 큰 장점입니다.
## 결정 트리: 스무고개로 예측하기

"소득이 5 이상인가? → 위도가 37.9 이하인가? → …" 질문을 따라 내려가 도착한 칸의 평균값으로 답합니다. 직선으로 표현할 수 없는 관계(위치에 따른 집값 등)를 잡아냅니다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

small = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(13, 5)); plot_tree(small, feature_names=list(X.columns), filled=True, fontsize=9); plt.show()

**코드 읽기**

- `DecisionTreeRegressor(max_depth=2)` — 질문을 최대 2단계만 하는 아주 얕은 트리. 왜 2인가: 그림으로 읽을 수 있을 만큼 작게 만들어 **트리가 어떻게 판단하는지** 보려는 것입니다. 진짜 성능용이 아닙니다.
- 트리는 어떻게 질문을 고르나: 모든 특징의 모든 경계값을 시도해 보고, 두 쪽으로 나눴을 때 **오차가 가장 많이 줄어드는** 질문을 고릅니다. 그림의 첫 질문이 `MedInc`(소득)인 것은 소득으로 나누는 것이 집값을 가장 잘 갈랐다는 뜻입니다.
- `plot_tree(model, feature_names=..., filled=True)` — 트리를 그림으로 그려 줍니다. `feature_names`를 주지 않으면 `x[0]` 같은 번호로 나와 읽을 수 없습니다. `filled=True`는 예측값이 클수록 진한 색으로 칠합니다. 각 상자의 `value`가 그 칸에 도착했을 때의 예측값(평균 집값), `samples`가 그 칸에 속한 학습 데이터 수입니다.
- 트리 모델에는 `StandardScaler`가 없습니다. "소득 > 5인가?"라는 질문은 값의 단위를 바꿔도 같은 질문이기 때문입니다. 전처리가 덜 필요하다는 점도 표 데이터에서 트리가 사랑받는 이유입니다.

깊이를 제한하지 않으면 트리는 학습 데이터를 **완벽하게 외울 때까지** 가지를 칩니다. 앞 레슨에서 본 과적합입니다.

In [ ]:
evaluate("트리 (깊이 제한 없음)", DecisionTreeRegressor(random_state=0))   # train MAE가 0에 가깝습니다
evaluate("트리 (깊이 8)", DecisionTreeRegressor(max_depth=8, random_state=0))

**코드 읽기**

- `max_depth`를 주지 않으면 잎 하나에 샘플 하나가 남을 때까지 쪼갭니다. train MAE가 0.000인 것이 그 증거입니다. `max_depth=8`은 "질문을 8번까지만"이라는 **복잡도 제한**이고, 앞 레슨의 정규화 표에서 "모델 크기 제한"에 해당합니다. `min_samples_leaf`(잎에 최소 몇 개는 남겨라)도 같은 목적의 손잡이입니다.
- 트리에 `random_state`가 있는 이유: 오차 감소량이 똑같은 질문이 여럿일 때 무작위로 고르기 때문입니다. 고정하지 않으면 실행마다 조금씩 다른 트리가 나옵니다.
## 앙상블: 약한 모델 여럿이 강한 모델 하나를 이긴다

- **랜덤 포레스트:** 데이터와 특징을 무작위로 달리해 트리를 수백 그루 만들고 **평균**을 냅니다. 개별 트리의 과적합이 서로 상쇄됩니다.
- **그래디언트 부스팅:** 트리를 하나씩 차례로 추가하되, 새 트리는 **지금까지의 오차**를 맞히도록 학습합니다. 표 데이터 대회의 단골 우승 모델입니다(XGBoost, LightGBM이 이 계열).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

forest = evaluate("랜덤 포레스트", RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0))
boost = evaluate("그래디언트 부스팅", HistGradientBoostingRegressor(random_state=0))
pd.DataFrame(results).T.round(3)

**코드 읽기**

- `RandomForestRegressor(n_estimators=100)` — 트리 100그루. 각 트리는 학습 데이터에서 **중복을 허용해 무작위로 뽑은** 샘플로, 그리고 질문할 때마다 **특징 일부만** 후보로 보고 자랍니다. 그래서 그루마다 다르게 틀리고, 평균을 내면 서로 상쇄됩니다. 개별 트리는 깊이 제한 없이 과적합시켜도 됩니다(그래서 train MAE가 낮음). 평균이 그것을 고쳐 줍니다.
- `n_jobs=-1` — 트리 100그루는 서로 독립이라 CPU 코어 전부를 써서 동시에 만들 수 있습니다. `-1`은 "가능한 코어를 다 써라"입니다.
- `HistGradientBoostingRegressor` — 그래디언트 부스팅의 빠른 구현(특징 값을 256개 구간으로 묶어서 계산). 트리를 한 그루씩 **차례로** 추가하되, 새 트리는 "지금까지의 예측이 틀린 만큼"을 맞히도록 학습합니다. 이름의 "그래디언트"는 2부에서 배울 경사하강법과 같은 뜻입니다: 오차가 줄어드는 방향으로 한 걸음씩. 랜덤 포레스트보다 보통 조금 더 정확하지만, 학습률·트리 수 같은 하이퍼파라미터에 더 민감하고 과적합도 더 쉽게 일어납니다. `XGBoost`, `LightGBM`, `CatBoost`가 같은 계열의 유명한 라이브러리입니다.
- 왜 두 앙상블을 다 보여주나: 실무에서 표 데이터 문제를 받으면 이 둘이 **첫 번째 진짜 후보**이기 때문입니다. 랜덤 포레스트는 손대지 않아도 무난하고, 부스팅은 튜닝하면 더 강합니다.
- `pd.DataFrame(results).T.round(3)` — 딕셔너리를 표로 바꾸고(`T`로 모델이 행이 되게 뒤집음) 소수점 3자리로 반올림. 노트북에서 셀의 마지막 줄에 표를 두면 자동으로 예쁘게 출력됩니다.
## 모델은 무엇을 보고 판단했을까

특징 하나의 값을 무작위로 섞었을 때 성능이 얼마나 떨어지는지 봅니다(permutation importance). 많이 떨어질수록 중요한 특징입니다.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(boost, X_test, y_test, n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=X.columns).sort_values().plot.barh(); plt.xlabel("drop in R2 when shuffled"); plt.show()

plt.scatter(y_test, boost.predict(X_test), s=3, alpha=.3); plt.plot([0, 5], [0, 5], "r")
plt.xlabel("true"); plt.ylabel("predicted"); plt.show()    # 대각선에 가까울수록 정확

**코드 읽기**

- `permutation_importance(model, X_test, y_test, n_repeats=5)` — 특징 하나의 열을 무작위로 뒤섞어(그 특징만 쓸모없게 만들어) 점수가 얼마나 떨어지는지 잽니다. 왜 이 방법인가: 트리 모델에는 `feature_importances_`라는 내장 속성도 있지만, 값의 종류가 많은 특징을 과대평가하는 편향이 있습니다. 뒤섞기 방식은 **어떤 모델에든** 같은 기준으로 쓸 수 있고 "실제 예측에 얼마나 기여하는가"를 직접 잽니다. `n_repeats=5`는 뒤섞기의 우연을 줄이려고 5번 반복해 평균합니다.
- 왜 `X_test`로 재나: 학습 데이터로 재면 외워 버린 특징까지 중요해 보입니다. 새 데이터에서 실제로 도움이 되는 특징을 알고 싶으므로 test로 잽니다.
- `.sort_values().plot.barh()` — pandas Series는 정렬과 막대그래프를 메서드 하나로 그립니다. `barh`(가로 막대)는 특징 이름이 길 때 읽기 좋습니다.
- 마지막 산점도(정답 vs 예측) — 지표 하나로는 안 보이는 것이 보입니다. 집값 5.0에서 점들이 세로로 늘어선 것은 데이터 수집 때 잘린 값 때문이고, 이런 오차는 모델을 바꿔도 줄지 않습니다. **지표 → 그림 → 데이터**로 돌아가 원인을 찾는 순서를 기억하세요.
## 핵심 정리

- 표 데이터는 pandas로 **먼저 들여다보고**, 기준선(평균으로 찍기, 선형 모델)부터 세웁니다.
- 회귀의 평가 지표는 MAE, R² 등. 분류의 정확도에 해당합니다.
- 결정 트리는 제한 없이 키우면 학습 데이터를 외웁니다(과적합).
- 여러 트리를 평균(랜덤 포레스트)하거나 차례로 오차를 보정(부스팅)하면 훨씬 강해집니다.
- 표 데이터 → 트리 앙상블, 이미지·텍스트 → 딥러닝이 기본 선택입니다.

## 스스로 점검

답을 머릿속으로 먼저 말해 본 뒤 펼쳐 보세요.

<details><summary>Q1. 깊이 제한 없는 트리의 train MAE가 거의 0인데 test MAE는 선형 회귀와 비슷합니다. 무슨 뜻인가요?</summary>

학습 데이터의 각 행을 사실상 통째로 외운 과적합 상태입니다. train 성능은 모델 실력을 말해 주지 않습니다.

</details>

<details><summary>Q2. 랜덤 포레스트에서 트리마다 데이터와 특징을 무작위로 다르게 주는 이유는?</summary>

모든 트리가 똑같으면 평균을 내도 그대로입니다. 서로 **다르게 틀리는** 트리들을 만들어야 평균을 냈을 때 오차가 상쇄됩니다.

</details>

<details><summary>Q3. 새 프로젝트에서 복잡한 모델부터 만들지 않고 기준선부터 세우는 이유는?</summary>

① 복잡한 모델의 점수가 좋은 것인지 판단할 기준이 생기고, ② 데이터 로딩·분할·평가 코드의 버그를 단순한 상황에서 먼저 잡을 수 있으며, ③ 단순한 모델로 충분한 경우도 많기 때문입니다.

</details>

## 직접 고쳐보기

1. `max_depth`를 2, 4, 8, 16, None으로 바꿔 train/test MAE를 표로 만들어 보세요. 어디서부터 과적합인가요?
2. `RandomForestRegressor`의 `n_estimators`를 1, 10, 100, 300으로 바꿔 보세요. 성능은 어디서 포화되나요?
3. 위도·경도 열을 빼고(`X.drop(columns=["Latitude", "Longitude"])`) 학습하면 얼마나 나빠지나요? 선형 회귀와 부스팅 중 어느 쪽이 더 크게 나빠지나요? 왜일까요?
4. 새 특징을 만들어 넣어 보세요(예: `df["AveRooms"] / df["AveOccup"]`). 이런 작업을 특징 공학(feature engineering)이라고 합니다. 딥러닝은 이 작업을 모델이 스스로 하게 만든 것입니다.
5. (도전) `sklearn.datasets.load_breast_cancer`(분류 문제)에 같은 모델들의 `Classifier` 버전을 적용해 비교표를 만들어 보세요.

<details><summary>힌트와 예상 결과 — 먼저 스스로 해 본 뒤 펼치세요</summary>

1. 깊이 2: train·test 모두 높음(과소적합). 8: test MAE 최소 근처(약 0.47). 16: train은 계속 내려가지만 test는 오히려 오르기 시작. None: train 0, test 0.47. 깊이 10~12 근처부터 과적합입니다.
2. 1그루 약 0.47(단일 트리와 같음), 10그루 약 0.37, 100그루 약 0.335, 300그루 약 0.33. 100 근처에서 포화합니다. 트리 수는 많을수록 좋지만 어느 순간부터 시간만 늘어납니다.
3. 부스팅이 더 크게 나빠집니다(R² 0.84 → 0.7대). 위치는 집값을 결정하는 비선형 정보이고, 트리는 그것을 "이 좌표 범위면 비싸다"로 잘 활용하지만 선형 모델은 원래 위도·경도를 직선적으로만 쓰고 있어 잃는 것이 적습니다.
4. `AveRooms / AveOccup`(사람당 방 수)을 넣으면 선형 회귀의 R²가 조금 오릅니다(0.59 → 0.6대). 트리 모델은 이런 비율을 스스로 근사할 수 있어 이득이 작습니다. 특징 공학은 단순한 모델에서 효과가 큽니다.
5. `RandomForestClassifier`, `HistGradientBoostingClassifier`, `LogisticRegression`(스케일러 포함) 모두 정확도 0.95~0.98. 회귀 → 분류로 바뀌어도 `evaluate` 함수의 지표만 `accuracy_score`로 바꾸면 됩니다.

</details>